# Federal Register — querying the new dataset

This notebook walks through `federal_register.parquet`, the dataset added to the Spicy Regs ETL in April 2026. It's sourced directly from [federalregister.gov's API v1](https://www.federalregister.gov/developers/api/v1) (not Mirrulations), and includes every Rule, Proposed Rule, Notice, and Presidential Document published since 2000-01-01.

## Why this dataset matters

The existing `dockets.parquet` / `documents.parquet` / `comments.parquet` come from regulations.gov — they tell you what agencies *posted for public comment* and what the public *said back*. The Federal Register is the official daily publication that *actually announces* a rule. Pairing the two unlocks:

- **Find the official rule for a docket you're analyzing** — `docket_ids_json` links FR documents back to regulations.gov dockets.
- **Search across notices and presidential documents** that never get a comment docket.
- **Track CFR section changes** via `cfr_references_json`.

## Schema at a glance

All columns are `VARCHAR` (matching the pipeline's all-Utf8 convention). Multi-valued nested fields are JSON-encoded — use DuckDB's `json_extract` / `unnest` to expand them. See [src/spicy_regs/pipeline/README.md](../src/spicy_regs/pipeline/README.md) for the full field reference.

## Setup

By default this notebook reads from R2. If you've just run the ETL locally and haven't uploaded yet, set `FR_PARQUET` to your local file.

In [1]:
import duckdb
from pathlib import Path

R2_BASE_URL = "https://r2.spicy-regs.dev"

# Default: read from R2. Override to a local path if you've run the ETL locally
# but haven't uploaded yet, e.g. FR_PARQUET = '../output/federal_register.parquet'.
LOCAL_FALLBACK = Path("../output/federal_register.parquet")
FR_PARQUET = (
    str(LOCAL_FALLBACK.resolve()) if LOCAL_FALLBACK.exists()
    else f"{R2_BASE_URL}/federal_register.parquet"
)

conn = duckdb.connect()
conn.execute("INSTALL httpfs; LOAD httpfs;")
print(f"reading from: {FR_PARQUET}")

reading from: /home/ekim/code/civictechdc/spicy-regs/output/federal_register.parquet


## Sanity check — schema and row count

DuckDB streams the Parquet file from R2 and only fetches the metadata footer for `DESCRIBE` and `COUNT(*)`. No full download.

In [2]:
conn.execute(f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT document_number) AS unique_docs,
        MIN(publication_date) AS earliest,
        MAX(publication_date) AS latest
    FROM read_parquet('{FR_PARQUET}')
""").fetchdf()

,rows,unique_docs,earliest,latest
0,793289,793289,2000-01-03,2026-04-27


In [3]:
# Schema
conn.execute(f"DESCRIBE SELECT * FROM read_parquet('{FR_PARQUET}')").fetchdf()

,column_name,column_type,null,key,default,extra
0,document_number,VARCHAR,YES,None,None,None
1,title,VARCHAR,YES,None,None,None
2,abstract,VARCHAR,YES,None,None,None
3,document_type,VARCHAR,YES,None,None,None
4,publication_date,VARCHAR,YES,None,None,None
5,effective_on,VARCHAR,YES,None,None,None
6,comments_close_on,VARCHAR,YES,None,None,None
7,signing_date,VARCHAR,YES,None,None,None
8,agencies_json,VARCHAR,YES,None,None,None
9,agency_slugs,VARCHAR,YES,None,None,None


## Counts by document type

The four FR document categories. Notices typically dominate — they include meeting announcements, agency reports, etc.

In [4]:
conn.execute(f"""
    SELECT document_type, COUNT(*) AS n
    FROM read_parquet('{FR_PARQUET}')
    GROUP BY document_type
    ORDER BY n DESC
""").fetchdf()

,document_type,n
0,Notice,631781
1,Rule,94090
2,Proposed Rule,60265
3,Presidential Document,7153


## Recent EPA proposed rules

The `agency_slugs` column is a comma-joined list of FR's agency slugs (e.g. `environmental-protection-agency`, `food-and-drug-administration`). Use `LIKE` to match a single agency since a single FR document can be issued by multiple agencies (joint rulemakings).

In [5]:
conn.execute(f"""
    SELECT
        publication_date,
        document_number,
        title,
        comments_close_on,
        html_url
    FROM read_parquet('{FR_PARQUET}')
    WHERE document_type = 'Proposed Rule'
      AND agency_slugs LIKE '%environmental-protection-agency%'
    ORDER BY publication_date DESC
    LIMIT 10
""").fetchdf()

,publication_date,document_number,title,comments_close_on,html_url
0,2026-04-24,2026-08012,Significant New Use Rules on Certain Chemical ...,2026-05-26,https://www.federalregister.gov/documents/2026...
1,2026-04-23,2026-07906,Air Plan Approval; Missouri; Control of Emissi...,2026-05-26,https://www.federalregister.gov/documents/2026...
2,2026-04-23,2026-07899,Hazardous Waste Management System; Identificat...,2026-05-20,https://www.federalregister.gov/documents/2026...
3,2026-04-23,2026-07904,Utah; Uinta Basin; 2015 8-Hour Ozone National ...,2026-05-26,https://www.federalregister.gov/documents/2026...
4,2026-04-23,2026-07894,Kentucky; Approval and Promulgation of State P...,2026-05-26,https://www.federalregister.gov/documents/2026...
5,2026-04-23,2026-07905,Air Plan Approval; Michigan; 2015 Ozone Modera...,2026-05-26,https://www.federalregister.gov/documents/2026...
6,2026-04-22,2026-07800,National Emission Standards for Hazardous Air ...,2026-06-22,https://www.federalregister.gov/documents/2026...
7,2026-04-16,2026-07405,Air Plan Revisions; Arizona; Maricopa County A...,2026-05-18,https://www.federalregister.gov/documents/2026...
8,2026-04-13,2026-07061,Hazardous and Solid Waste Management System: D...,2026-06-12,https://www.federalregister.gov/documents/2026...
9,2026-04-10,2026-06938,Air Plan Approval; New York; Interstate Transp...,2026-05-11,https://www.federalregister.gov/documents/2026...


## Executive Orders — `subtype` + `executive_order_number`

Presidential documents have `document_type = 'Presidential Document'` and a `subtype` like `Executive Order`, `Memorandum`, `Proclamation`, etc. Executive orders also carry a numeric `executive_order_number`.

In [6]:
conn.execute(f"""
    SELECT
        signing_date,
        executive_order_number AS eo_number,
        title,
        html_url
    FROM read_parquet('{FR_PARQUET}')
    WHERE subtype = 'Executive Order'
      AND signing_date >= '2024-01-01'
    ORDER BY signing_date DESC
    LIMIT 10
""").fetchdf()

,signing_date,eo_number,title,html_url
0,2026-04-18,14401,Accelerating Medical Treatments for Serious Me...,https://www.federalregister.gov/documents/2026...
1,2026-04-03,14400,Urgent National Action To Save College Sports,https://www.federalregister.gov/documents/2026...
2,2026-03-31,14399,Ensuring Citizenship Verification and Integrit...,https://www.federalregister.gov/documents/2026...
3,2026-03-26,14398,Addressing DEI Discrimination by Federal Contr...,https://www.federalregister.gov/documents/2026...
4,2026-03-24,14397,Further Continuance of the Federal Emergency M...,https://www.federalregister.gov/documents/2026...
5,2026-03-20,14396,Preserving America's Game,https://www.federalregister.gov/documents/2026...
6,2026-03-16,14395,Establishing the Task Force To Eliminate Fraud,https://www.federalregister.gov/documents/2026...
7,2026-03-13,14393,Promoting Access to Mortgage Credit,https://www.federalregister.gov/documents/2026...
8,2026-03-13,14392,Ensuring Truthful Advertising of Products Clai...,https://www.federalregister.gov/documents/2026...
9,2026-03-13,14391,Adjusting Certain Delegations Under the Defens...,https://www.federalregister.gov/documents/2026...


## The cross-link: join FR documents to regulations.gov dockets

`docket_ids_json` is a JSON array of regulations.gov docket IDs (e.g. `["EPA-HQ-OW-2021-0736", "FRL-8885-01-OW"]`). DuckDB's `json_extract` + `unnest` expands one row per `(fr_doc, docket_id)` pair so we can join to `dockets.parquet`.

The query below finds 2024 EPA proposed rules that have a matching docket on regulations.gov, plus the docket title from the Mirrulations side.

In [7]:
DOCKETS_PARQUET = f"{R2_BASE_URL}/dockets.parquet"

conn.execute(f"""
    WITH fr_with_dockets AS (
        SELECT
            document_number,
            publication_date,
            title AS fr_title,
            html_url,
            -- Expand docket_ids_json into one row per linked docket.
            unnest(
                cast(json_extract(docket_ids_json, '$') AS VARCHAR[])
            ) AS linked_docket_id
        FROM read_parquet('{FR_PARQUET}')
        WHERE document_type = 'Proposed Rule'
          AND publication_date >= '2024-01-01'
          AND agency_slugs LIKE '%environmental-protection-agency%'
          AND docket_ids_json IS NOT NULL
    )
    SELECT
        fr.publication_date,
        fr.document_number AS fr_doc,
        -- linked_docket_id arrives JSON-quoted from json_extract; strip the quotes.
        TRIM(fr.linked_docket_id, '\"') AS docket_id,
        d.title AS docket_title,
        fr.html_url
    FROM fr_with_dockets fr
    LEFT JOIN read_parquet('{DOCKETS_PARQUET}') d
        ON d.docket_id = TRIM(fr.linked_docket_id, '\"')
    WHERE d.docket_id IS NOT NULL
    ORDER BY fr.publication_date DESC
    LIMIT 10
""").fetchdf()

,publication_date,fr_doc,docket_id,docket_title,html_url
0,2026-04-24,2026-08012,EPA-HQ-OPPT-2025-2932,Significant New Use Rules on Certain Chemical ...,https://www.federalregister.gov/documents/2026...
1,2026-04-23,2026-07899,EPA-R06-RCRA-2026-2641,Hazardous Waste Management System; Identificat...,https://www.federalregister.gov/documents/2026...
2,2026-04-23,2026-07905,EPA-R05-OAR-2025-0235,Western Michigan 2015 Ozone Moderate NOx RACT ...,https://www.federalregister.gov/documents/2026...
3,2026-04-23,2026-07905,EPA-R05-OAR-2024-0137,West Michigan VOC RACT,https://www.federalregister.gov/documents/2026...
4,2026-04-23,2026-07906,EPA-R07-OAR-2025-3822,Air Plan Approval; Missouri; Control of Emissi...,https://www.federalregister.gov/documents/2026...
5,2026-04-23,2026-07904,EPA-R08-OAR-2024-0001,UB 2nd Extension and DAAD,https://www.federalregister.gov/documents/2026...
6,2026-04-22,2026-07800,EPA-HQ-OAR-2025-1348,National Emission Standards for Hazardous Air ...,https://www.federalregister.gov/documents/2026...
7,2026-04-16,2026-07405,EPA-R09-OAR-2026-1684,Air Plan Revisions; Arizona; Maricopa County A...,https://www.federalregister.gov/documents/2026...
8,2026-04-13,2026-07061,EPA-HQ-OLEM-2020-0107,Hazardous and Solid Waste Management System: D...,https://www.federalregister.gov/documents/2026...
9,2026-04-10,2026-06940,EPA-R05-OAR-2025-0237,Ohio Source-Specific RACT for Cleveland Ohio,https://www.federalregister.gov/documents/2026...


## Going further — comment counts via `comments_index.parquet`

Once you have `(fr_doc, docket_id)` pairs, you can pull comment counts from `comments_index.parquet` (the tiny metadata file used by [search_capabilities.ipynb](search_capabilities.ipynb)) and answer **"how much public response did this rule generate?"** without touching the 24M-row comments table.

In [8]:
conn.execute(f"""
    WITH fr_dockets AS (
        SELECT
            document_number,
            publication_date,
            title AS fr_title,
            html_url,
            TRIM(unnest(
                cast(json_extract(docket_ids_json, '$') AS VARCHAR[])
            ), '\"') AS linked_docket
        FROM read_parquet('{FR_PARQUET}')
        WHERE document_type = 'Proposed Rule'
          AND publication_date >= '2023-01-01'
          AND docket_ids_json IS NOT NULL
    ),
    comment_counts AS (
        SELECT docket_id, SUM(row_count) AS n_comments
        FROM read_parquet('{R2_BASE_URL}/comments_index.parquet')
        GROUP BY docket_id
    )
    SELECT
        fr.publication_date,
        fr.document_number AS fr_doc,
        fr.linked_docket,
        cc.n_comments,
        fr.fr_title
    FROM fr_dockets fr
    JOIN comment_counts cc ON cc.docket_id = fr.linked_docket
    ORDER BY cc.n_comments DESC
    LIMIT 15
""").fetchdf()

,publication_date,fr_doc,linked_docket,n_comments,fr_title
0,2023-02-07,2023-02102,FNS-2022-0043,287922.0,Child Nutrition Programs: Revisions to Meal Pa...
1,2023-07-31,2023-15405,CEQ-2023-0003,245946.0,National Environmental Policy Act Implementing...
2,2023-02-01,2023-00610,EERE-2014-BT-STD-0005,84813.0,Energy Conservation Program: Energy Conservati...
3,2023-08-02,2023-16475,EERE-2014-BT-STD-0005,84813.0,Energy Conservation Program: Energy Conservati...
4,2024-02-14,2024-02007,EERE-2014-BT-STD-0005,84813.0,Energy Conservation Program: Energy Conservati...
5,2023-02-28,C1-2023-00610,EERE-2014-BT-STD-0005,84813.0,Energy Conservation Program: Energy Conservati...
6,2023-02-28,2023-03864,EERE-2014-BT-STD-0005,84813.0,Energy Conservation Program: Energy Conservati...
7,2023-08-17,2023-16515,NHTSA-2023-0022,63088.0,Corporate Average Fuel Economy Standards for P...
8,2023-08-25,2023-18309,NHTSA-2023-0022,63088.0,Public Hearing for Corporate Average Fuel Econ...
9,2023-08-25,2023-18310,NHTSA-2023-0022,63088.0,Corporate Average Fuel Economy Standards for P...


## Searching the FR text directly

Two layers, mirroring [search_capabilities.ipynb](search_capabilities.ipynb):

### Layer A — DuckDB `ILIKE` over the parquet (recommended)

Fast on the parquet because DuckDB streams + pushes down predicates. Works for any column.

In [9]:
conn.execute(f"""
    SELECT publication_date, document_type, document_number, title
    FROM read_parquet('{FR_PARQUET}')
    WHERE (title ILIKE '%PFAS%' OR abstract ILIKE '%PFAS%')
      AND publication_date >= '2024-01-01'
    ORDER BY publication_date DESC
    LIMIT 10
""").fetchdf()

,publication_date,document_type,document_number,title
0,2026-04-16,Notice,2026-07402,"Tucson International Airport Area Site, Tucson..."
1,2026-04-13,Rule,2026-07062,Modification to the Start of the Submission Pe...
2,2026-04-06,Proposed Rule,2026-06662,Drinking Water Contaminant Candidate List 6-Draft
3,2026-02-27,Rule,2026-03944,Implementing Statutory Addition of Certain Per...
4,2025-12-18,Notice,2025-23311,Response to Comments for the Department of Vet...
5,2025-11-13,Proposed Rule,2025-19882,Perfluoroalkyl and Polyfluoroalkyl Substances ...
6,2025-07-10,Notice,2025-12889,National Drinking Water Advisory Council; Meeting
7,2025-05-13,Rule,2025-08168,Perfluoroalkyl and Polyfluoroalkyl Substances ...
8,2025-01-21,Proposed Rule,2024-29239,Clean Water Act Methods Update Rule 22 for the...
9,2025-01-17,Proposed Rule,2024-31406,Toxics Release Inventory (TRI); Clarification ...


### Layer B — prebuilt search index

`federal_register_search.json.gz` is the FR equivalent of `docket_search.json.gz` — gzipped JSON with abbreviated keys (`id`, `a`, `t`, `x`, `d`, `s`, `u`) for client-side fuzzy/prefix search.

**Heads up on size:** the full backfill projects to ~100 MB gzipped (vs 5–10 MB for dockets). Loading it all into a browser is not viable — it's emitted for use cases like local notebooks, server-side filtering, or a future scoped-by-year build. The cell below caches it locally so re-running the notebook doesn't re-fetch.

In [10]:
import gzip
import json
import time
import urllib.error
import urllib.request

INDEX_URL = f"{R2_BASE_URL}/federal_register_search.json.gz"
CACHE_PATH = Path(".cache/federal_register_search.json.gz")

def _fetch_with_retry(url: str, attempts: int = 4) -> bytes:
    req = urllib.request.Request(url, headers={"User-Agent": "spicy-regs-notebook"})
    for i in range(attempts):
        try:
            with urllib.request.urlopen(req) as resp:
                return resp.read()
        except urllib.error.HTTPError as e:
            if e.code != 429 or i == attempts - 1:
                raise
            wait = 2 ** i
            print(f"  429 rate-limited, retrying in {wait}s…")
            time.sleep(wait)
    raise RuntimeError("unreachable")

if CACHE_PATH.exists():
    raw = CACHE_PATH.read_bytes()
    print(f"using cached index at {CACHE_PATH}")
else:
    print(f"fetching {INDEX_URL}…")
    raw = _fetch_with_retry(INDEX_URL)
    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    CACHE_PATH.write_bytes(raw)

if raw[:2] == b"\x1f\x8b":
    raw = gzip.decompress(raw)
fr_index = json.loads(raw)

print(f"index version: {fr_index['version']}")
print(f"docs indexed: {fr_index['count']:,}")
fr_index['docs'][0]

fetching https://r2.spicy-regs.dev/federal_register_search.json.gz…


HTTPError: HTTP Error 404: Not Found

In [ ]:
def search_fr_index(query: str, agency_slug: str | None = None, doc_type: str | None = None, limit: int = 10):
    """Substring search over the prebuilt FR index.

    Field map: id=document_number, a=agency_slugs, t=title, x=document_type,
    d=publication_date, s=abstract, u=html_url.
    """
    q = query.lower()
    hits = []
    for doc in fr_index['docs']:
        if agency_slug and agency_slug not in doc.get('a', ''):
            continue
        if doc_type and doc.get('x') != doc_type:
            continue
        if q in doc.get('t', '').lower() or q in doc.get('s', '').lower():
            hits.append(doc)
            if len(hits) >= limit:
                break
    return hits

for hit in search_fr_index("climate", doc_type="Rule", limit=5):
    print(f"[{hit['d']}] {hit['id']} — {hit['t'][:90]}")

## Tips

- **`agency_slugs` is FR's slug, not regulations.gov's agency code.** FR uses `environmental-protection-agency`; regulations.gov uses `EPA`. They don't match directly. Use `agency_slugs LIKE '%slug%'` for FR and join through `docket_ids_json` if you need to bridge to the regulations.gov side.
- **Multi-agency joint rulemakings.** A single FR document can list 2+ agencies. The `agency_slugs` column comma-joins them. If you need first-class multi-row access, `unnest(string_split(agency_slugs, ','))`.
- **`publication_date` is a string, not a date.** All columns are `VARCHAR` to match the pipeline's all-Utf8 staging convention. ISO-8601 dates sort lexicographically, so `WHERE publication_date >= '2024-01-01'` works fine. Cast with `CAST(publication_date AS DATE)` if you need date arithmetic.
- **Corrections.** FR publishes correction notices that get a separate `document_number` (e.g. `C1-2024-11192` corrects `2024-11192`). The merge step keeps the latest by `publication_date`.
- **Where's the body text?** We don't ingest the full document body — only metadata + abstract. Hit `body_html_url` directly if you need the full text of a single document.